# Exotic model-grid adapters

`pyramids.grids` regrids model grids that are **not** plain rasters into a regular-grid
`Dataset`:

- `from_orca` — curvilinear `(ny, nx)` lon/lat ocean grids (NEMO/ORCA),
- `from_octahedral` — octahedral reduced-Gaussian fields (ECMWF `O` grids),
- `from_healpix` — HEALPix sphere pixelizations (RING or NESTED).

Each adapter reshapes its grid into an existing pyramids regridding bridge
(`mesh_to_grid` or `grid_points`) and returns a single-band `Dataset`. No external data
or extra dependencies are needed to run this notebook.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from pyramids.grids import from_healpix, from_octahedral, from_orca


def show(ds, title):
    """Quick look at a regridded Dataset (no-data masked)."""
    arr = ds.read_array().astype(float)
    nodata = ds.no_data_value[0]
    data = np.ma.masked_equal(arr, nodata) if nodata is not None else arr
    minx, miny, maxx, maxy = (float(v) for v in ds.bbox)
    fig, ax = plt.subplots(figsize=(7, 3.2))
    im = ax.imshow(data, extent=[minx, maxx, miny, maxy], origin="upper")
    ax.set_title(title)
    ax.set_xlabel("longitude")
    ax.set_ylabel("latitude")
    fig.colorbar(im, ax=ax, shrink=0.85)
    return fig

## ORCA curvilinear grid

ORCA stores coordinates as two `(ny, nx)` arrays of longitudes and latitudes describing
quadrilateral cells. `from_orca` stitches them into a UGRID quad mesh and interpolates it
to a regular grid. Here we fake a curvilinear grid by warping a regular lon/lat mesh.

In [ ]:
ny, nx = 60, 120
lon = np.linspace(-180.0, 180.0, nx)
lat = np.linspace(-80.0, 80.0, ny)
lon2d, lat2d = np.meshgrid(lon, lat)
lon2d = lon2d + 6.0 * np.sin(np.radians(lat2d))  # curvilinear warp
data2d = np.cos(np.radians(lat2d)) * np.sin(np.radians(lon2d))

orca_ds = from_orca(lon2d, lat2d, data2d, cell_size=2.0)
print(
    "rows, columns, bands, epsg:",
    orca_ds.rows,
    orca_ds.columns,
    orca_ds.band_count,
    orca_ds.epsg,
)

In [ ]:
# NBVAL_IGNORE_OUTPUT
show(orca_ds, "from_orca — curvilinear ORCA field")

## Octahedral reduced-Gaussian grid

An octahedral grid is a single ragged sequence of points with known lat/lon. `from_octahedral`
wraps them as scattered points and interpolates with `gdal.Grid`. We stand in for the real
grid with deterministic pseudo-random samples.

In [ ]:
rng = np.random.default_rng(0)
n_points = 4000
lons = rng.uniform(-180.0, 180.0, n_points)
lats = rng.uniform(-80.0, 80.0, n_points)
values = np.cos(np.radians(lats)) * np.sin(np.radians(lons))

octa_ds = from_octahedral(lats, lons, values, cell_size=3.0, algorithm="nearest")
print(
    "rows, columns, bands, epsg:",
    octa_ds.rows,
    octa_ds.columns,
    octa_ds.band_count,
    octa_ds.epsg,
)

In [ ]:
# NBVAL_IGNORE_OUTPUT
show(octa_ds, "from_octahedral — scattered reduced-Gaussian samples")

## HEALPix grid (RING and NESTED)

HEALPix stores one value per pixel, indexed by `nside` (so `12 * nside**2` pixels).
`from_healpix` computes each pixel's centre lon/lat in plain NumPy — no `healpy` — and supports
both the **RING** and **NESTED** pixel orderings. The same value array under the two orderings
describes different fields, because the orderings number the pixels differently.

In [ ]:
nside = 16
npix = 12 * nside**2
values = np.cos(np.linspace(0.0, 6.0 * np.pi, npix))

ring_ds = from_healpix(values, nside=nside, nest=False, cell_size=2.0)
nest_ds = from_healpix(values, nside=nside, nest=True, cell_size=2.0)
print("RING :", ring_ds.rows, ring_ds.columns, ring_ds.band_count)
print("NESTED:", nest_ds.rows, nest_ds.columns, nest_ds.band_count)

In [ ]:
# NBVAL_IGNORE_OUTPUT
show(ring_ds, "from_healpix — RING ordering")

In [ ]:
# NBVAL_IGNORE_OUTPUT
show(nest_ds, "from_healpix — NESTED ordering")

`nside` is derived automatically from the value count when omitted, and invalid inputs raise a
clear error (e.g. a length that is not `12 * nside**2`, or `nest=True` with a non-power-of-two
`nside`).

In [ ]:
try:
    from_healpix(np.zeros(10), cell_size=2.0)
except ValueError as exc:
    print("rejected:", exc)